# Budgerigar：AutoMachine 层级 Token 记忆 Smoke Training

第一代 GRU 已学会时序行为但未保存可区分句子内容。本模型用 16 层固定 token bank、逐 tick 优化器和全 bank 注意力读取替代单一隐状态，并加入正确/打乱目标对比损失。仍然没有监听、结束或朗读状态机。

In [ ]:
#@title 1. 更新项目并安装训练依赖
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [key for key in list(sys.modules) if key=='budgerigar' or key.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. 挂载 Drive 并复用特征与统计
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
TARGET_SPEAKER='arctic_slt' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
STATS_PATH=WORK_ROOT/'features'/f'stats.smoke64.{TARGET_SPEAKER}.{FEATURE_FINGERPRINT}.pt'
assert FEATURE_MANIFEST.is_file(),FEATURE_MANIFEST
assert STATS_PATH.is_file(),'请先完成 NeuralEcho notebook 第 4 步的 smoke 统计'
import torch,json
stats=torch.load(STATS_PATH,map_location='cpu',weights_only=True)
print(FEATURE_MANIFEST,STATS_PATH)

In [ ]:
#@title 3. 单个 episode 前向结构检查
from budgerigar.echo_data import load_pairs,EchoEpisodeDataset
from budgerigar.hierarchical_echo import HierarchicalEchoConfig,create_hierarchical_echo
pairs=[pair for pair in load_pairs(FEATURE_MANIFEST,TARGET_SPEAKER) if pair.split=='train']
preview=EchoEpisodeDataset(pairs[:1],stats,thinking_frames=(16,28),preload=True)
inputs,outputs,voice,metadata=preview[0]
model=create_hierarchical_echo(HierarchicalEchoConfig(token_slots=16))
with torch.no_grad(): predicted,strength,memory,diagnostics=model(inputs[:64].unsqueeze(0))
print('input/output:',inputs.shape,outputs.shape,'preview:',predicted.shape,'tokens:',memory[0].shape)
assert predicted.shape[-1]==100 and memory[0].shape[1]==16

In [ ]:
#@title 4. T4 层级记忆 smoke training
MAX_STEPS=200 #@param {type:'integer'}
BATCH_SIZE=4 #@param {type:'integer'}
TOKEN_SLOTS=16 #@param {type:'integer'}
if not torch.cuda.is_available(): raise RuntimeError('请选择 GPU runtime')
from budgerigar.train_hierarchical import HierarchicalTrainingConfig,train_hierarchical_echo
RUN_DIR=WORK_ROOT/'checkpoints'/f'hierarchical_echo_{TARGET_SPEAKER}_{FEATURE_FINGERPRINT}'
training=HierarchicalTrainingConfig(target_speaker=TARGET_SPEAKER,batch_size=BATCH_SIZE,max_steps=MAX_STEPS)
model_config=HierarchicalEchoConfig(token_slots=TOKEN_SLOTS)
report=train_hierarchical_echo(FEATURE_MANIFEST,RUN_DIR,training,model_config,stats=stats)
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 5. 保存运行元数据
from budgerigar.experiment import write_run_metadata
metadata=write_run_metadata(RUN_DIR/'run_metadata.json',FEATURE_MANIFEST,{'architecture':'hierarchical_token_echo','token_slots':TOKEN_SLOTS,'best_validation_content_loss':report['best_validation_content_loss']},repository=REPO_DIR)
print(metadata.read_text(encoding='utf-8'))